# validate_ghsa
Bronze GHSA Raw → Valid / Reject 분리.
- **Fatal** (Reject): 식별자(`ghsa_id`/`id`)가 아예 없음 → 이후 어떤 테이블의 키도 만들 수 없으므로 폐기 불가 원칙의 예외로 Reject 처리.
- **Warn** (Valid로 유지, 플래그만 기록): CVSS/설명/CWE 결측 — 데이터가 불완전하다고 무조건 버리지 않는다(§5.1).
- Reject에는 `raw_payload` 원본을 그대로 보관해 재처리 가능하게 한다.

In [ ]:
%run 00_silver_common

In [ ]:
def read_json_flex(path: str):
    """GHSA Bronze가 JSON 배열(멀티라인)로 오는지 JSONL(줄 단위)로 오는지 몰라도 되게 방어적으로 읽는다."""
    try:
        df = spark.read.option("multiLine", "true").json(path)
        if len(df.columns) > 0:
            return df
    except Exception:
        pass
    return spark.read.json(path)

raw_df = read_json_flex(f"{BRONZE_GHSA_PATH}/{snapshot_date}/*")
raw_source_cols = raw_df.columns


In [ ]:
identifier_candidate = coalesce_existing(raw_df, ["ghsa_id", "id"])
raw_df = raw_df.withColumn("_identifier_candidate", identifier_candidate)

fatal_cond = F.col("_identifier_candidate").isNull()

warn_flags = flags_array(
    F.when(existing_col(raw_df, "severity").isNull(), F.lit("MISSING_SEVERITY")),
    F.when(coalesce_existing(raw_df, ["description", "details"]).isNull(), F.lit("MISSING_DESCRIPTION")),
    F.when(F.size(F.coalesce(existing_array_col(raw_df, "cwes"), F.array())) == 0, F.lit("MISSING_CWE")),
)
raw_df = raw_df.withColumn("_validation_flags", warn_flags)


In [ ]:
reject_df = (
    raw_df.filter(fatal_cond)
    .withColumn("reject_reason", F.lit("MISSING_IDENTIFIER"))
    .withColumn("raw_payload", F.to_json(F.struct(*raw_source_cols)))
    .withColumn("source", F.lit("ghsa"))
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("ingested_at", F.lit(INGESTED_AT))
    .select("raw_payload", "reject_reason", "source", "batch_id", "ingested_at")
)

valid_df = (
    raw_df.filter(~fatal_cond)
    .withColumnRenamed("_validation_flags", "validation_flags")
    .drop("_identifier_candidate")
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("ingested_at", F.lit(INGESTED_AT))
)

reject_count = reject_df.count()
valid_count = valid_df.count()


In [ ]:
if reject_count > 0:
    write_append(reject_df, TBL["reject_ghsa"])

write_overwrite(valid_df, TBL["stg_ghsa_valid"])

total = valid_count + reject_count
reject_rate = reject_count / total if total > 0 else 0.0
log_batch_run("validate_ghsa", "SUCCESS", record_count=valid_count)
log_dq_metric("validate_ghsa", "reject_rate", reject_rate, passed=True,
              detail=f"valid={valid_count}, reject={reject_count}")

print(f"[validate_ghsa] valid={valid_count} reject={reject_count} batch_id={batch_id}")
